# 🦁 Mossad-pentast-ai v2 — Master Colab Pro Training Pipeline
### Base Model: `Qwen/Qwen3.5-9B` | Offensive Dataset: 31,213 Unique 7-Phase Scenarios
### Auto-Sync: Google Drive (`/content/drive/MyDrive/mossad_pentast_v2_lora`)
### Live Agent Heartbeat: Auto-Reports to Antigravity Manager

This notebook automatically trains **Mossad-pentast-ai v2** on Google Colab Pro. All progress is automatically broadcasted to the local AI manager and saved permanently to Google Drive.

In [ ]:
# 1. Mount Google Drive for Permanent Zero-Loss Checkpointing
import os, sys
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/mossad_pentast_v2_lora'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'[✓] Google Drive Connected! Permanent Storage: {SAVE_DIR}')

In [ ]:
# 2. Install High-Performance ML Dependencies
!pip install -q -U transformers peft accelerate bitsandbytes datasets pandas pyarrow

In [ ]:
# 3. Hardware Auto-Detection & Adaptive Hyperparameters
import torch

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0

print(f'🔥 Active GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)')

if 'A100' in gpu_name:
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    COMPUTE_DTYPE = torch.bfloat16
    MAX_SEQ_LEN = 1024
    print('[*] Profile: Ultra-Fast A100 High-Throughput Mode (BF16, seq=1024)')
elif 'L4' in gpu_name or 'V100' in gpu_name:
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
    COMPUTE_DTYPE = torch.bfloat16 if 'L4' in gpu_name else torch.float16
    MAX_SEQ_LEN = 896
    print(f'[*] Profile: High-Performance {gpu_name} Mode (seq=896)')
else:
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
    COMPUTE_DTYPE = torch.float16
    MAX_SEQ_LEN = 384
    print('[*] Profile: Standard T4 Memory-Optimized Mode (FP16, seq=384)')

In [ ]:
# 4. Fetch 31,213 Unique Offensive Scenarios + Anti-Degradation Multi-Task Mixture
import json, urllib.request, random

GITHUB_BASE = 'https://raw.githubusercontent.com/mmossad2124-blip/BugBounty-Copilot/main/data_pipeline/processed_datasets'
DATASET_FILES = [
    'elite_bugbounty_cot.jsonl',
    'legendary_milestones_curated.jsonl',
    'tier2_32_sources_arsenal.jsonl',
    'stage1_web_cloud_arsenal.jsonl',
    'stage2_kernel_apt_arsenal.jsonl',
    'master_redteam_cot.jsonl',
]

SEVEN_PHASE_SYSTEM = """You are Mossad-pentast-ai (v2 - Qwen3.5-9B), an Elite Black-Box Red Team, Zero-Day Vulnerability Researcher, and Autonomous Bug Bounty AI Agent. For every target or vulnerability task, apply the full 7-Phase Offensive Methodology:
1. Reconnaissance & OSINT
2. Scanning & Attack Surface Enumeration
3. Vulnerability Discovery & Root-Cause (CWE-ID / MITRE ATT&CK) Mapping
4. Exploitation & Proof-of-Concept (PoC) Verification
5. Post-Exploitation & Privilege Escalation Analysis
6. Vulnerability Chaining & Lateral Movement
7. Full Attack-Path Synthesis & Next Proactive Step Proposal."""

def upgrade_to_7phase_persona(messages):
    user_txt, asst_txt = '', ''
    for m in messages:
        if m['role'] == 'user':
            user_txt = m['content']
        elif m['role'] == 'assistant':
            asst_txt = m['content']
    user_txt = user_txt.replace(
        'Provide root-cause analysis, exploit methodology, and defensive verification.',
        'Execute a full 7-Phase Black-Box Red Team & Bug Bounty technical analysis, including Root-Cause (CWE), PoC methodology, Privilege Escalation, and Chaining.'
    )
    if '#### 4. Remediation & Defensive Hardening' in asst_txt:
        parts = asst_txt.split('#### 4. Remediation & Defensive Hardening')
        asst_txt = (
            parts[0].strip()
            + '\n\n#### 4. Post-Exploitation, Privilege Escalation & Chaining (Phases 5-7)\n'
            + '- **Privilege Escalation**: Leverage initial access primitives to inspect internal tokens, service configurations, and trust boundaries for vertical escalation.\n'
            + '- **Vulnerability Chaining & Lateral Movement**: Chain the identified flaw with internal endpoints, SSRF/IDOR primitives, or session artifacts to maximize Bug Bounty impact.\n'
            + '- **Next Proactive Action**: Enumerate adjacent API routes and test parameter desynchronization across authenticated endpoints.'
        )
    return [
        {'role': 'system', 'content': SEVEN_PHASE_SYSTEM},
        {'role': 'user', 'content': user_txt},
        {'role': 'assistant', 'content': asst_txt},
    ]

raw_samples = []
for fname in DATASET_FILES:
    url = f'{GITHUB_BASE}/{fname}'
    print(f'[*] Fetching dataset source: {fname} ...')
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=60) as r:
            for line in r.read().decode('utf-8', errors='ignore').splitlines():
                if line.strip():
                    try:
                        obj = json.loads(line.strip())
                        if 'messages' in obj and len(obj['messages']) >= 3:
                            raw_samples.append(obj['messages'])
                    except Exception:
                        pass
    except Exception as e:
        print(f'    [!] Warning: {e}')

seen = set()
unique_convos = []
for msgs in raw_samples:
    u = msgs[1]['content'] if len(msgs) > 1 else ''
    if u and u not in seen:
        seen.add(u)
        unique_convos.append(upgrade_to_7phase_persona(msgs))

print(f'[✓] Total Unique Red Team Scenarios Loaded: {len(unique_convos)}')

# Add Anti-Degradation Balanced Mixture (CWE-ID Triage + CoT Exploit Reasoning)
cwe_triage = [
    [{'role': 'system', 'content': 'You are a vulnerability researcher and CWE classification expert.'},
     {'role': 'user', 'content': 'Analyze the following CVE description and map it to the single most appropriate CWE ID.\nRespond ONLY with the CWE ID in the format: CWE-XXX\n\nCVE Description: In the Linux kernel through 6.7.1, there is a use-after-free in cec_queue_msg_fh.'},
     {'role': 'assistant', 'content': 'CWE-416'}],
    [{'role': 'system', 'content': 'You are a vulnerability researcher and CWE classification expert.'},
     {'role': 'user', 'content': 'Analyze the following CVE description and map it to the single most appropriate CWE ID.\nRespond ONLY with the CWE ID in the format: CWE-XXX\n\nCVE Description: A reflected cross-site scripting vulnerability in user profile search field.'},
     {'role': 'assistant', 'content': 'CWE-79'}],
    [{'role': 'system', 'content': 'You are a vulnerability researcher and CWE classification expert.'},
     {'role': 'user', 'content': 'Analyze the following CVE description and map it to the single most appropriate CWE ID.\nRespond ONLY with the CWE ID in the format: CWE-XXX\n\nCVE Description: Server-side request forgery leading to AWS IMDSv1 token exposure.'},
     {'role': 'assistant', 'content': 'CWE-918'}],
    [{'role': 'system', 'content': 'You are a vulnerability researcher and CWE classification expert.'},
     {'role': 'user', 'content': 'Analyze the following CVE description and map it to the single most appropriate CWE ID.\nRespond ONLY with the CWE ID in the format: CWE-XXX\n\nCVE Description: Out-of-bounds write heap corruption in libxml2 parse function.'},
     {'role': 'assistant', 'content': 'CWE-787'}],
    [{'role': 'system', 'content': 'You are a vulnerability researcher and CWE classification expert.'},
     {'role': 'user', 'content': 'Analyze the following CVE description and map it to the single most appropriate CWE ID.\nRespond ONLY with the CWE ID in the format: CWE-XXX\n\nCVE Description: SQL injection in product search filter allows arbitrary database read via UNION SELECT.'},
     {'role': 'assistant', 'content': 'CWE-89'}],
] * 200

mcq_reasoning = [
    [{'role': 'system', 'content': 'You are a cybersecurity and offensive exploitation expert.'},
     {'role': 'user', 'content': 'Question: In x86-64 Linux binary exploitation, which register stores the first integer argument passed to a function call?\nOptions: A) RAX, B) RDI, C) RSI, D) RDX\n\nProvide ONLY the final answer in the format: ANSWER: X'},
     {'role': 'assistant', 'content': '<think>\nIn x86-64 System V AMD64 ABI calling convention, function arguments are passed in RDI, RSI, RDX, RCX, R8, R9. The first integer argument is in RDI.\n</think>\nANSWER: B'}],
    [{'role': 'system', 'content': 'You are a cybersecurity and offensive exploitation expert.'},
     {'role': 'user', 'content': 'Question: Which technique allows an attacker to bypass Data Execution Prevention (DEP / NX) by executing existing code sequences ending in RET?\nOptions: A) NOP Sled, B) Return-Oriented Programming (ROP), C) Heap Spraying, D) Format String Attack\n\nProvide ONLY the final answer in the format: ANSWER: X'},
     {'role': 'assistant', 'content': '<think>\nReturn-Oriented Programming (ROP) stitches together small instruction sequences ending in ret (gadgets) already present in executable memory to bypass W^X / NX.\n</think>\nANSWER: B'}],
] * 200

all_training_samples = unique_convos + cwe_triage + mcq_reasoning
random.seed(42)
random.shuffle(all_training_samples)
print(f'[✓] Final Balanced Dataset Ready: {len(all_training_samples)} total conversations ({len(unique_convos)} Red Team + {len(cwe_triage)} CWE Triage + {len(mcq_reasoning)} Reasoning)')

In [ ]:
# 5. Load Base Model (Qwen/Qwen3.5-9B) & Setup LoRA Adapter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

BASE_MODEL_ID = 'Qwen/Qwen3.5-9B'
print(f'[*] Loading Base Model {BASE_MODEL_ID} in 4-bit NF4...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True
    )
except Exception:
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True
    )

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False})

# Check if Google Drive already has weights to resume from
adapter_path = os.path.join(SAVE_DIR, 'adapter_model.safetensors')
if os.path.exists(adapter_path):
    print(f'[✓] FOUND EXISTING LORA CHECKPOINT IN GOOGLE DRIVE ({SAVE_DIR})! Resuming on top...')
    model = PeftModel.from_pretrained(model, SAVE_DIR, is_trainable=True)
else:
    print('[*] Initializing fresh Elite QLoRA Adapter (r=16, alpha=32)...')
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.05,
        bias='none'
    )
    model = get_peft_model(model, lora_config)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.print_trainable_parameters()

In [ ]:
# 6. Continuous Training Loop with Live Progress & Google Drive Auto-Sync
import time

TOTAL_STEPS = 1200
LR = 1.2e-4
CHECKPOINT_INTERVAL = 50
REPORT_INTERVAL = 25

STATUS_FILE = os.path.join(SAVE_DIR, 'live_status.json')

def record_live_status(step, total, loss, avg_loss, elapsed_sec, eta_sec, status='training'):
    data = {
        'status': status,
        'step': step,
        'total_steps': total,
        'loss': round(loss, 4),
        'avg_loss': round(avg_loss, 4),
        'gpu': gpu_name,
        'gpu_mem_used_gb': round(torch.cuda.memory_allocated() / (1024**3), 2) if torch.cuda.is_available() else 0,
        'elapsed_minutes': round(elapsed_sec / 60, 1),
        'eta_minutes': round(eta_sec / 60, 1),
        'last_ping': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
    }
    try:
        with open(STATUS_FILE, 'w') as f:
            json.dump(data, f, indent=2)
    except Exception:
        pass

import gc
gc.collect()
torch.cuda.empty_cache()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=30, num_training_steps=TOTAL_STEPS)

model.train()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
optimizer.zero_grad()

print(f'[*] 🔥 STARTING MASTER TRAINING: {TOTAL_STEPS} Steps | Auto-reporting live to Agent...')
t_start = time.time()
step_idx = 0
sample_ptr = 0
running_loss = 0.0

while step_idx < TOTAL_STEPS:
    accum_loss = 0.0
    for _ in range(GRAD_ACCUM):
        convo = all_training_samples[sample_ptr % len(all_training_samples)]
        sample_ptr += 1
        try:
            text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        except Exception:
            text = '\n'.join([f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>" for m in convo])

        enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
        input_ids = enc['input_ids']
        attention_mask = enc['attention_mask']
        labels = input_ids.clone()

        with torch.amp.autocast(device_type='cuda', dtype=COMPUTE_DTYPE):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        accum_loss += loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()
    torch.cuda.empty_cache()

    step_idx += 1
    running_loss += accum_loss

    if step_idx % REPORT_INTERVAL == 0 or step_idx == 1:
        elapsed = time.time() - t_start
        avg_l = running_loss / (REPORT_INTERVAL if step_idx > 1 else 1)
        running_loss = 0.0
        eta = (elapsed / step_idx) * (TOTAL_STEPS - step_idx)
        print(f'  [Step {step_idx:4d}/{TOTAL_STEPS}] Loss: {accum_loss:.4f} (Avg: {avg_l:.4f}) | LR: {scheduler.get_last_lr()[0]:.2e} | Elapsed: {elapsed/60:.1f}m | ETA: {eta/60:.1f}m')
        # Live record status
        record_live_status(step_idx, TOTAL_STEPS, accum_loss, avg_l, elapsed, eta, status='training')

    # Auto-save directly to Google Drive
    if step_idx % CHECKPOINT_INTERVAL == 0 or step_idx == TOTAL_STEPS:
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        progress_meta = {
            'step': step_idx,
            'total_steps': TOTAL_STEPS,
            'samples_covered': sample_ptr,
            'loss': accum_loss,
            'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
        }
        with open(os.path.join(SAVE_DIR, 'training_progress.json'), 'w') as f:
            json.dump(progress_meta, f, indent=2)
        print(f'  💾 [AUTO-SYNC] Successfully synced latest LoRA weights to Google Drive: {SAVE_DIR}')

    record_live_status(TOTAL_STEPS, TOTAL_STEPS, 0.0, 0.0, time.time() - t_start, 0.0, status='completed')
print(f'\n🎉 TRAINING FULLY COMPLETED IN {(time.time() - t_start)/60:.1f} MINUTES!')
print(f'🏆 Final Compounded LoRA Weights Saved at: {SAVE_DIR}')

In [ ]:
# 7. Live 7-Phase Red Team Verification Test
print('='*75)
print('🎯 TESTING TRAINED MOSSAD-PENTAST-AI V2 LIVE ON A RECON/EXPLOIT TASK')
print('='*75)

test_prompt = [
    {'role': 'system', 'content': SEVEN_PHASE_SYSTEM},
    {'role': 'user', 'content': 'Target endpoint: https://api.enterprise-banking.com/v1/auth/exchange\nAnalyze parameter reflection, SSRF potential to AWS metadata service (IMDSv2 vs IMDSv1), and provide the full 7-Phase technical attack synthesis.'}
]

test_text = tokenizer.apply_chat_template(test_prompt, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(test_text, return_tensors='pt').to(model.device)

model.eval()
with torch.no_grad():
    gen_tokens = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(gen_tokens[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)